# Day 3 — Feature Engineering & Multi-Modal Representation

Stage-2 (row-preserving) feature engineering: turn the Day-2 cleaned dataset
(`kickstarter_processed.csv`, 331,464 × 12) into a leakage-free, ML-ready
**sparse** feature matrix.

This notebook demonstrates the phases built in `src/preprocessing/` and
`src/features/` and assembled into a single `FeaturePipeline`. See
`docs/day3_roadmap.md` for the frozen spec.

**Pipeline (Phase L order):** FeatureBuilder → [numeric: log→cap→scale |
categorical: encode | text: preprocess→TF-IDF] → fuse → select.

In [1]:
import sys, os

# Run from the repo root so data/ and artifacts/ paths resolve consistently
# with the module defaults (guarded so re-running the cell is idempotent).
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())

import warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)  # quiet the per-stage INFO/UserWarning noise

import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)

## Phase 0 — Train/Test Split (the leakage foundation)

Split *before* fitting anything. Stratified on the target so the ~40% positive
rate is preserved in both partitions; indices persisted for reproducibility.

In [2]:
from src.features.splitting import DataSplitter

df = pd.read_csv("data/processed/kickstarter_processed.csv")
split = DataSplitter().split(df)   # persists artifacts/splits/*

print(f"train: {split.X_train.shape[0]:,} rows | test: {split.X_test.shape[0]:,} rows")
print(f"pos-rate  train={split.y_train.mean():.4f}  test={split.y_test.mean():.4f}  "
      f"(overall {df['target'].mean():.4f})")
split.X_train.head(3)

train: 265,171 rows | test: 66,293 rows
pos-rate  train=0.4038  test=0.4038  (overall 0.4038)


,name,category,main_category,currency,deadline,goal,launched,country,usd_goal_real,name_clean
56697,A Winter Storm - A Novel,Fiction,Publishing,USD,2012-12-28,6000.0,2012-11-28 00:24:14,US,6000.00,a winter storm a novel
266870,PilaFoot,Technology,Technology,EUR,2017-12-29,37250.0,2017-11-29 08:34:40,BE,44673.91,pilafoot
314310,JB Couture Cupcakes Hits the Shore!,Food,Food,USD,2014-05-14,28000.0,2014-04-19 02:45:56,US,28000.00,jb couture cupcakes hits the shore


## Phase A — Feature Profiling (train-only)

Measure distributions and emit a candidate-transformation list. Deciding the
transforms from *train only* is itself part of the leakage discipline.

In [3]:
from src.features.profiling import FeatureProfiler

profile = FeatureProfiler().profile(split.X_train, split.y_train, X_test=split.X_test)
pd.DataFrame(
    [(r.feature, r.transform, r.reason) for r in profile.recommendations],
    columns=["feature", "transform", "reason"],
)

,feature,transform,reason
0,goal,log1p,heavy right skew (73.1) collapses to -0.12 und...
1,usd_goal_real,log1p,heavy right skew (80.9) collapses to -0.14 und...
2,category,frequency_encode,high cardinality (159 > 25); one-hot would exp...
3,main_category,one_hot,low cardinality (15 <= 25)
4,currency,one_hot,low cardinality (14 <= 25)
5,country,one_hot,low cardinality (22 <= 25)
6,goal|usd_goal_real,drop_one_of_pair,near-collinear (pearson 0.957); keep one
7,name_clean,tfidf_small_vocab,short titles (~5.6 words); cap TF-IDF features...
8,campaign_duration,engineer_from_dates,derive (deadline - launched).days + temporal p...
9,goal_per_day,guard_divide_by_zero,74 row(s) have duration <= 0; goal/duration wo...


## Phases B–L — The full `FeaturePipeline`

One object fits every stage on train and transforms any partition. `fit` is the
only method that calls each sub-stage's `fit`; `transform` only transforms — so
once fitted on train, applying it to test cannot leak.

In [4]:
from src.features.pipeline import FeaturePipeline

pipe = FeaturePipeline().fit(split.X_train, split.y_train)
X_train = pipe.transform(split.X_train)
X_test = pipe.transform(split.X_test)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"final feature count: {len(pipe.get_feature_names_out())}")
print(f"format: {type(X_train).__name__}  |  density: "
      f"{X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100:.3f}%")

X_train: (265171, 865)  |  X_test: (66293, 865)
final feature count: 865
format: csr_matrix  |  density: 2.212%


### Sparse vs dense — why the matrix must stay sparse

The fused matrix is >97% zeros (the 800-column TF-IDF block dominates). Storing
it dense would be orders of magnitude larger.

In [5]:
n, d = X_train.shape
dense_mb = n * d * 8 / 1e6
sparse_mb = (X_train.data.nbytes + X_train.indices.nbytes + X_train.indptr.nbytes) / 1e6
print(f"dense : {dense_mb:,.0f} MB")
print(f"sparse: {sparse_mb:,.1f} MB  ({dense_mb / sparse_mb:.0f}x smaller)")

dense : 1,835 MB
sparse: 61.9 MB  (30x smaller)


### Which features actually matter? (mutual information)

From the pipeline metadata: the top features by mutual information with the
target. The tabular features dominate — "multi-modal" here is weak title text
plus a strong tabular block.

In [6]:
meta = pipe.build_metadata()
top = pd.DataFrame(meta["selection"]["top_features"], columns=["feature", "mutual_info"])


def origin(f):
    if f in pipe.numeric_features or f.endswith("_freq") or f.startswith(
        ("main_category_", "currency_", "country_")
    ):
        return "tabular"
    return "text"


top["origin"] = top["feature"].apply(origin)
top

,feature,mutual_info,origin
0,usd_goal_real,0.162274,tabular
1,goal_per_day,0.110936,tabular
2,category_freq,0.051394,tabular
3,campaign_duration,0.021316,tabular
4,new,0.021147,text
5,album,0.020714,text
6,uppercase_ratio,0.017342,tabular
7,film,0.015923,text
8,book,0.015355,text
9,project,0.013641,text


## Persistence — model-ready artifacts

The fitted pipeline is part of the model: it carries the learned TF-IDF
vocabulary, scaler statistics, and selection mask. Saved as one artifact plus a
JSON output contract.

In [7]:
from scipy import sparse as sp

pipe.save()  # artifacts/feature_pipeline/{feature_pipeline.joblib, feature_metadata.json}
sp.save_npz("artifacts/fused_features/X_train_selected.npz", X_train)
sp.save_npz("artifacts/fused_features/X_test_selected.npz", X_test)
np.save("artifacts/fused_features/y_train.npy", split.y_train.to_numpy())
np.save("artifacts/fused_features/y_test.npy", split.y_test.to_numpy())

print("Saved:")
print("  artifacts/feature_pipeline/feature_pipeline.joblib")
print("  artifacts/feature_pipeline/feature_metadata.json")
print(f"  artifacts/fused_features/X_train_selected.npz  {X_train.shape}")
print(f"  artifacts/fused_features/X_test_selected.npz   {X_test.shape}")

Saved:
  artifacts/feature_pipeline/feature_pipeline.joblib
  artifacts/feature_pipeline/feature_metadata.json
  artifacts/fused_features/X_train_selected.npz  (265171, 865)
  artifacts/fused_features/X_test_selected.npz   (66293, 865)


## Reproducibility check

A reloaded pipeline transforms the test set identically — the fitted state
round-trips, and inference on a single campaign works (production path).

In [8]:
reloaded = FeaturePipeline.load("artifacts/feature_pipeline/feature_pipeline.joblib")
X_test_reload = reloaded.transform(split.X_test)
print(f"reloaded transform identical: {(X_test != X_test_reload).nnz == 0}")

one_campaign = reloaded.transform(split.X_test.iloc[[0]])
print(f"single-campaign inference:    {one_campaign.shape}")

reloaded transform identical: True
single-campaign inference:    (1, 865)


## Summary

| Stage | Output |
|---|---|
| Split | 265,171 train / 66,293 test, stratified |
| Numeric branch | log → cap → scale (14 features) |
| Categorical | one-hot + frequency (52 features, sparse) |
| Text | TF-IDF, 1–2 grams, 800 features (sparse) |
| Fusion | `scipy.sparse.hstack` → 866 features |
| Selection | drop correlated `goal` → **865 features** |

Output: a leakage-free, sparse, model-ready matrix. **Next (Day 4):** logistic
regression and tree models on these 865 features.